# RQ1 — Table 1: paper-reported versus reproduced baselines

This notebook compares only the four methods shared by the source paper and the group: LightGBM, GAT, GraphSAGE/SAGE, and Meta-IFD. Group plus variants are deliberately excluded.

Paper values are transcribed from Table IV (PDF page 10). Group values are recomputed from five runs on the fixed 553-node paper test split.

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from IPython.display import display

EXPECTED_MANIFEST_SHA256 = 'd2fa5a9ca3563d7cfc477ec980cc98c90eff4f301eca65c818a608e44c70e480'
EXPECTED_DATA_SHA256 = '8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def search_roots():
    roots = [Path.cwd(), Path('/kaggle/input'), Path('/kaggle/working'), Path(r'D:/Takedo/Documents/DAP/rq1_reporting_tables_20260926')]
    return [p for p in roots if p.exists()]

def locate_by_hash(name, expected_hash):
    matches = []
    for root in search_roots():
        for path in root.rglob(name):
            try:
                if sha256_file(path) == expected_hash:
                    matches.append(path)
            except OSError:
                pass
    unique = {str(p.resolve()): p for p in matches}
    assert unique, f'Cannot find {name} with SHA-256 {expected_hash}. Upload the complete reporting bundle.'
    return next(iter(unique.values()))

manifest_path = locate_by_hash('table2_source_manifest.json', EXPECTED_MANIFEST_SHA256)
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['source_data_sha256'] == EXPECTED_DATA_SHA256
source_dir = manifest_path.parent
print('Paper source:', manifest['paper']['table'], 'PDF page', manifest['paper']['pdf_page'])

Paper source: Table IV PDF page 10


In [2]:
expected = {item['name']: item['sha256'] for item in manifest['files']}
expected[manifest['paper']['values_file']] = manifest['paper']['values_file_sha256']
for name, digest in expected.items():
    path = source_dir / name
    assert path.is_file(), path
    assert sha256_file(path) == digest, f'SHA-256 mismatch: {path}'
print(f'Integrity check passed for {len(expected)} source files.')

Integrity check passed for 9 source files.


In [3]:
model_sources = {
    'LightGBM': 'lightgbm_paper_split_test_results.csv',
    'GAT': 'gat_paper_split_test_results.csv',
    'SAGE': 'graphsage_paper_split_test_results.csv',
    'Meta-IFD': 'metaifd_paper_split_test_results.csv',
}
metric_columns = {
    'Precision': 'test_precision', 'Recall': 'test_recall',
    'Binary-F1': 'test_binary_f1', 'Micro-F1': 'test_micro_f1', 'Macro-F1': 'test_macro_f1',
}
paper = pd.read_csv(source_dir / manifest['paper']['values_file']).set_index('model')
rows = []
for model, filename in model_sources.items():
    p = paper.loc[model]
    paper_row = {'Model': model, 'Source': 'Paper reported'}
    for label, stem in [('Precision', 'precision'), ('Recall', 'recall'), ('Binary-F1', 'binary_f1'), ('Micro-F1', 'micro_f1'), ('Macro-F1', 'macro_f1')]:
        paper_row[f'{label} mean'] = float(p[f'{stem}_mean'])
        paper_row[f'{label} SD'] = float(p[f'{stem}_sd'])
    paper_row['Delta Macro-F1 vs paper (pp)'] = 0.0
    rows.append(paper_row)

    df = pd.read_csv(source_dir / filename)
    base = df[df['variant'] == 'base']
    assert len(base) == 5 and set(base['seed'].astype(int)) == {12, 13, 14, 15, 16}
    assert ((base[['test_tn', 'test_fp', 'test_fn', 'test_tp']].sum(axis=1)) == 553).all()
    reproduced_name = {'GAT': 'GAT-Norm', 'SAGE': 'GraphSAGE-Norm'}.get(model, model)
    reproduced = {'Model': reproduced_name, 'Source': 'Reproduced baseline'}
    for label, column in metric_columns.items():
        values = base[column].to_numpy(dtype=float) * 100
        reproduced[f'{label} mean'] = values.mean()
        reproduced[f'{label} SD'] = values.std(ddof=1)
    reproduced['Delta Macro-F1 vs paper (pp)'] = reproduced['Macro-F1 mean'] - paper_row['Macro-F1 mean']
    rows.append(reproduced)
table_numeric = pd.DataFrame(rows)

In [4]:
def mean_sd(row, metric):
    return f"{row[f'{metric} mean']:.2f} ± {row[f'{metric} SD']:.2f}"
table_display = table_numeric[['Model', 'Source']].copy()
for metric in ['Precision', 'Recall', 'Binary-F1', 'Micro-F1', 'Macro-F1']:
    table_display[metric] = table_numeric.apply(lambda r: mean_sd(r, metric), axis=1)
table_display['Δ Macro vs paper (pp)'] = table_numeric.apply(
    lambda r: '—' if r['Source'] == 'Paper reported' else f"{r['Delta Macro-F1 vs paper (pp)']:+.2f}", axis=1
)
display(table_display.style.hide(axis='index').set_caption(
    'Table 1. Paper-reported versus reproduced baseline results on HEIG-Phish (%)'
))

Model,Source,Precision,Recall,Binary-F1,Micro-F1,Macro-F1,Δ Macro vs paper (pp)
LightGBM,Paper reported,94.13 ± 0.67,97.01 ± 1.13,95.54 ± 0.37,96.06 ± 0.31,96.00 ± 0.32,—
LightGBM,Reproduced baseline,93.65 ± 0.00,97.93 ± 0.00,95.74 ± 0.00,96.20 ± 0.00,96.16 ± 0.00,+0.16
GAT,Paper reported,90.20 ± 1.52,95.19 ± 0.86,92.62 ± 0.95,93.38 ± 0.90,93.31 ± 0.90,—
GAT-Norm,Reproduced baseline,81.51 ± 3.56,92.12 ± 2.99,86.47 ± 2.87,87.41 ± 2.76,87.35 ± 2.76,-5.96
SAGE,Paper reported,93.46 ± 0.45,96.02 ± 0.62,94.72 ± 0.44,95.33 ± 0.39,95.27 ± 0.39,—
GraphSAGE-Norm,Reproduced baseline,90.68 ± 0.97,96.02 ± 1.08,93.27 ± 0.79,93.96 ± 0.71,93.90 ± 0.72,-1.37
Meta-IFD,Paper reported,97.37 ± 0.55,98.01 ± 0.17,97.68 ± 0.27,97.97 ± 0.24,97.94 ± 0.24,—
Meta-IFD,Reproduced baseline,97.19 ± 0.71,97.51 ± 0.59,97.35 ± 0.22,97.69 ± 0.20,97.65 ± 0.20,-0.29


### Interpretation guardrail

LightGBM and Meta-IFD have direct paper descriptions/code support; GAT and SAGE are group reimplementations because their full baseline implementations were not published. This table measures reproduction proximity and must not be used to attribute gains to UFE.

In [5]:
output_dir = Path('/kaggle/working/rq1_table_outputs/table1') if Path('/kaggle/working').exists() else Path.cwd() / 'rq1_table_outputs' / 'table1'
output_dir.mkdir(parents=True, exist_ok=True)
table_numeric.to_csv(output_dir / 'table1_paper_vs_reproduced_numeric.csv', index=False)
table_display.to_csv(output_dir / 'table1_paper_vs_reproduced_display.csv', index=False)
def dataframe_to_markdown(frame):
    clean = frame.astype(str).map(lambda x: x.replace('|', chr(92) + '|'))
    lines = ['| ' + ' | '.join(clean.columns) + ' |', '|' + '|'.join(['---'] * len(clean.columns)) + '|']
    lines.extend('| ' + ' | '.join(row) + ' |' for row in clean.to_numpy().tolist())
    return '\n'.join(lines) + '\n'
(output_dir / 'table1_paper_vs_reproduced.md').write_text(dataframe_to_markdown(table_display), encoding='utf-8')
(output_dir / 'table1_paper_vs_reproduced.tex').write_text(
    table_display.to_latex(index=False, escape=True, caption='Paper-reported versus reproduced baseline results on HEIG-Phish (percent).', label='tab:paper-reproduction'), encoding='utf-8'
)
metadata = {
    'table': 1, 'manifest_sha256': EXPECTED_MANIFEST_SHA256,
    'source_data_sha256': EXPECTED_DATA_SHA256,
    'paper_pdf_sha256': manifest['paper']['pdf_sha256'],
    'paper_source': f"{manifest['paper']['table']}, PDF page {manifest['paper']['pdf_page']}",
    'models': list(model_sources), 'model_seeds': [12, 13, 14, 15, 16],
}
(output_dir / 'table1_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved Table 1 artifacts to', output_dir)

Saved Table 1 artifacts to /kaggle/working/rq1_table_outputs/table1
